In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
path = './data/house_prices/'

In [3]:
def add_to_class(Class):
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

In [4]:
class KaggleHouse(Dataset):
    def __init__(self, train=True):
        super().__init__()
        self.train = train
        self.data = None
        self.raw_data = pd.read_csv(path + ('train.csv' if train else 'test.csv'))

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        get_tensor = lambda x: torch.tensor(x.values.astype(float), dtype=torch.float32)
        if self.train == True:
            label = 'SalePrice'
            features = get_tensor(self.data.drop(columns=[label]))
            labels = get_tensor(self.data[label])
            return features[idx], labels[idx]
        else:
            features = get_tensor(self.data)
            return features[idx]


In [5]:
train_dataset = KaggleHouse(train=True)
test_dataset = KaggleHouse(train=False)

In [6]:
train_dataset.raw_data.shape, test_dataset.raw_data.shape

In [7]:
def preprocess():
    # Remove the ID and label columns
    label = 'SalePrice'
    features = pd.concat(
        (train_dataset.raw_data.drop(columns=['Id', label]),
         test_dataset.raw_data.drop(columns=['Id'])))
    
    # Standardize numerical columns
    numeric_features = features.dtypes[features.dtypes!='object'].index
    features[numeric_features] = features[numeric_features].apply(lambda x: (x - x.mean()) / (x.std()))
    
    # Replace NAN numerical features by 0
    features[numeric_features] = features[numeric_features].fillna(0)

    # Replace discrete features by one-hot encoding
    features = pd.get_dummies(features, dummy_na=True)
    
    # Save preprocessed features
    train_dataset.data = features[:train_dataset.raw_data.shape[0]].copy()
    train_dataset.data[label] = train_dataset.raw_data[label]
    test_dataset.data = features[train_dataset.raw_data.shape[0]:].copy()
    print('Preprocessing complete')

In [8]:
preprocess()
print('Train shape:', train_dataset.data.shape)
print('Test shape:', test_dataset.data.shape)

In [9]:
display(train_dataset.data.head(2))
display(test_dataset.data.head(2))
print(train_dataset.data.shape, test_dataset.data.shape)

In [10]:
features = train_dataset.data.drop(columns=['SalePrice'])
label = train_dataset.data[['SalePrice']]
display(features.head(2))
display(label.head(2))

In [11]:
def convert_to_tensor(x):
    tensor = torch.tensor(x.values.astype(float), dtype=torch.float32)
    return tensor

In [12]:
features = convert_to_tensor(features)
label = convert_to_tensor(label)

In [13]:
# Loss function
def rmsle_loss(y_pred, y_true):
    epsilon = 1e-6
    loss = torch.sqrt(F.mse_loss(torch.log1p(y_pred.clamp(min=epsilon)), torch.log1p(y_true.clamp(min=epsilon))))
    return loss

In [14]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.LazyLinear(1)
        self.net.weight.data.normal_(0, 0.01)
        self.net.bias.data.fill_(0)

    def forward(self, X):
        return self.net(X)

In [15]:
folds = 5
skf = KFold(n_splits=folds, shuffle=True, random_state=42)

In [16]:
# Metrics
criterion = torch.nn.MSELoss()
batch_size = 64

In [17]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [18]:
epochs = 10
Loss = []
model = LinearRegression()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(epochs):
    epoch_loss = 0
    for i, (features, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(features).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    Loss.append(epoch_loss / len(train_loader))
    print(f'Epoch: {epoch+1}/{epochs}. Train loss: {Loss[-1]:.4f}.')

In [19]:
# Metrics
criterion = rmsle_loss
batch_size = 64

In [20]:
train_dataset.data["SalePrice"] = np.log1p(train_dataset.data["SalePrice"])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [21]:
epochs = 10
Loss = []
model_1 = LinearRegression()
optimizer = torch.optim.SGD(model_1.parameters(), lr=0.01)

for epoch in range(epochs):
    epoch_loss = 0
    for i, (features, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model_1(features).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    Loss.append(epoch_loss / len(train_loader))
    print(f'Epoch: {epoch+1}/{epochs}. Train loss: {Loss[-1]:.4f}.')

In [22]:
def predict(model, test_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for features in test_loader:
            outputs = model(features).view(-1)
            predictions.extend(outputs.numpy().flatten())
    return predictions

In [25]:
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [27]:
preds = predict(model, test_loader)
preds_1 = predict(model_1, test_loader)

In [28]:
preds_1 = torch.tensor(preds_1)
print(preds)
print(torch.expm1(preds_1).tolist())

In [ ]:
epochs = 10
Loss = []
models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(features, label)):
    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(train_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size)
    val_loader = DataLoader(val_subset, batch_size=batch_size)
    model = LinearRegression()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    print(f'FOLD: {fold+1}/{folds}')
    L = []
    
    for epoch in range(epochs):
        epoch_loss = 0
        for i, (features, labels) in enumerate(train_loader):
            optimizer.zero_grad()
            outputs = model(features).view(-1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        L.append(epoch_loss / len(train_loader))
        
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for features, labels in val_loader:
                outputs = model(features).view(-1)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        print(f'Epoch: {epoch+1}/{epochs}. Train loss: {L[-1]:.4f}. Val loss: {val_loss:.4f}')

    models.append(model)
    Loss.append(L)


In [ ]:
Loss[0]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, ax in enumerate(axes.flat):
    ax.plot(range(1, len(Loss[i]) + 1), Loss[i], label=f"Fold {i+1}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"Fold {i+1}")
    ax.legend()

plt.tight_layout()
plt.show()